# Building Linked Interactive Dashboards with Plotly & Dash

**Name:** Victor Lee
**GitHub repo:** "https://github.com/PamShinz1/Global-Development-Explorer"
**Live dashboard:** "https://global-development-explorer.onrender.com"
**Video walkthrough:** "https://drive.google.com/file/d/1ieGY_hh-azfaDxlPZA_EzmYstHMQz3F4/view?usp=sharing"

## 1. Visualization Techniques

This dashboard combines four complementary chart types, each answering a
different kind of question about the same underlying data:

| Chart | Question it answers | Why it's here |
|---|---|---|
| **Choropleth map** | "Where in the world is X high or low?" | Gives geographic context immediately — patterns by region jump out that a table never would. |
| **Bubble scatter plot** (GDP per capita vs. life expectancy, sized by population) | "Is there a relationship between two variables, and how does a third (population) play in?" 
| **Horizontal bar chart** | "Who are the top/bottom performers?" | Bar charts are still the best tool for ranking and direct magnitude comparison — something a map or scatter plot is bad at. |
| **Line chart** | "How has this one entity changed over time?" | Once a user has found something interesting (a country) via the other three views, the line chart answers the natural follow-up: *how did it get here?* |

We put them on one dashboard instead of four seperate plots because each view only tells part of the story. The map shows where, the scatter
plot shows relationships, the bar chart shows ranking, and the line
chart shows history. Shown separately, the audience has to hold each
view in their head to connect them. Linking them — a shared Year slider and
Continent filter, plus click-to-cross-filter — lets the reader ask a
question once ("what's happening in Asia in 1997?") and see the answer from
four angles simultaneously.


## 2. Visualization Library: Plotly + Dash

Dash is an open-source Python
framework for building web-based analytical dashboards, created and
maintained by **Plotly**.  Dash is Apache-2.0 licensed and free to use. 

**Installation.**
```bash
pip install dash plotly pandas
```

```python
app.run(jupyter_mode="inline", port=8060)
```

**Why Dash?**
- Dash was chosen here because 
  Plotly Express charts are interactive*by default,
  which is a big head start for a dashboard, and the model
  maps very directly onto the updated charts,
  which is exactly what we needd to demonstrate.

**Limitations.**
- Dash apps need a running Python server, they are not static HTML you
  can just email someone . This is why we need to deploy the app somewhere rather than just
  attach an HTML file.
- Multi-user deployment needs a production
  server rather than Dash's built-in development server


## 3. Demonstration

### 3.1 Dataset and Cleaning

We use the Gapminder dataset (life expectancy, population, and GDP per
capita for 142 countries, 1952-2007), chosen because it naturally supports
all four chart types and has a noticable pattern (the relationship between
wealth and health across the developing world) that's easy for a reader
unfamiliar with the data to follow.

The dataset bundled with Plotly Express, so
no download step or API key is required, this also keeps the dashboard
fully self-contained and reproducible.

Cleaning steps applied (see code cell below):
1. Rename columns to readable names (eg: `lifeExp` → `LifeExpectancy`).
2. Drop any rows with missing key fields .
3. Create a `PopulationMillions` column so bubble sizes in the scatter plot
   are on a more comprehensible scale.
4. Cast `Year` to `int`.

In [ ]:
import pandas as pd
import plotly.express as px

raw = px.data.gapminder()
print(raw.shape)
raw.head()

In [ ]:
df = raw.copy()
df = df.rename(columns={
    "country": "Country", "continent": "Continent", "year": "Year",
    "lifeExp": "LifeExpectancy", "pop": "Population",
    "gdpPercap": "GDPperCapita", "iso_alpha": "ISOAlpha", "iso_num": "ISONumeric",
})

before = len(df)
df = df.dropna(subset=["Country", "Continent", "Year", "LifeExpectancy",
                        "Population", "GDPperCapita"])
print(f"Dropped {before - len(df)} rows with missing values")

df["PopulationMillions"] = df["Population"] / 1_000_000
df["Year"] = df["Year"].astype(int)
df.head()

### 3.2 Building the Basic Components



In [ ]:
year = 2007
snapshot = df[df["Year"] == year]

fig_map = px.choropleth(
    snapshot, locations="ISOAlpha", color="LifeExpectancy",
    hover_name="Country", color_continuous_scale="Viridis",
    title=f"Life Expectancy by Country ({year})",
)
fig_map.show()

In [ ]:
fig_scatter = px.scatter(
    snapshot, x="GDPperCapita", y="LifeExpectancy",
    size="PopulationMillions", color="Continent",
    hover_name="Country", log_x=True, size_max=55,
    title=f"GDP per Capita vs. Life Expectancy ({year})",
)
fig_scatter.show()

In [ ]:
top15 = snapshot.nlargest(15, "GDPperCapita").sort_values("GDPperCapita")
fig_bar = px.bar(
    top15, x="GDPperCapita", y="Country", orientation="h", color="Continent",
    title=f"Top 15 Countries by GDP per Capita ({year})",
)
fig_bar.show()

In [ ]:
country = "United States"
trend = df[df["Country"] == country].sort_values("Year")
fig_line = px.line(
    trend, x="Year", y="LifeExpectancy", markers=True,
    title=f"Life Expectancy Trend: {country}",
)
fig_line.show()

### 3.3 Adding Interactivity: The Full Dash App

Now we combine all four charts into one Dash app.

## Components

- A **Year slider** and **Continent dropdown** that filter the map, scatter
  plot, and bar chart.
- **Click-to-cross-filter**: clicking a country on the map, in the scatter
  plot, or in the bar chart writes that country's name into a shared
  `dcc.Store`, which the line chart's callback listens to and redraws
  around.

The full source is in `app.py` in the repository. The same
code is reproduced here so the notebook is self-contained.

In [ ]:
%%writefile app.py



In [ ]:
# Run the dashboard directly inside this notebook
from app import app

app.run(jupyter_mode="inline", port=8060)